# Ghana TTS DataGen — synthetic TTS data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GhanaNLP/ghana-tts-datagen/blob/main/notebooks/ghana_tts_datagen.ipynb)

Generate synthetic TTS training audio from text, voice-cloning the built-in
male/female speakers with the Ghana NLP Community `ghana-tts-36k` model.
Output: `wavs/*.wav` + a manifest in your TTS format
(LJSpeech / Piper / VITS / MeloTTS) — ready for your trainer.

> **Use a GPU runtime** (required). Colab: `Runtime → Change runtime type → T4 GPU`.
> The model is private — set `HF_TOKEN` in Colab secrets or enter it below.

## Install

In [ ]:
!apt-get -qq install -y ffmpeg >/dev/null
!git clone -q https://github.com/GhanaNLP/ghana-tts-datagen.git
%cd ghana-tts-datagen
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — switch to a GPU runtime!'

## (Required) Hugging Face token
The model is **private**. You need a token with read access. Set it in Colab
secrets (`HF_TOKEN`), or enter it below.

In [ ]:
import os, getpass
_tok = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ').strip()
if _tok:
    os.environ['HF_TOKEN'] = _tok
    print('Token set.')
else:
    print('WARNING: No token set — model loading will fail!')

## Configure your run
Point at a text dataset on the Hub and its text column. (List
`ghananlpcommunity` datasets with
`!python -m ghana_tts_datagen --list-datasets`.)

In [ ]:
DATASET = "ghananlpcommunity/CHANGE-ME"   # your text dataset id (or use a text file below)
TEXT_COLUMN = "text"                        # the text column
# TEXT_FILE = "sentences.txt"               # alternative source: one sentence per line
NAME = "demo"                                # output → data/demo/
FORMATS = "ljspeech"                         # ljspeech | piper | vits | melo (comma list)
SR = 22050                                   # output sample rate (e.g. 24000 for MeloTTS)
PRECISION = "fp32"                           # fp16 = faster/½ VRAM on a T4 (check quality); bf16 needs A100/L4

## Preview a few clips
Hear the result before a big run.

In [ ]:
!python -m ghana_tts_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --name {NAME} --sample-rate {SR} --precision {PRECISION} --preview 3

import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'data/{NAME}/preview/*.wav')):
    print(w)
    display(Audio(w))

## Generate
Small demo run (0.2 h) — bump `--hours` for a real run. Re-run the same cell to
**resume** (finished rows are skipped).

**Speed on a GPU:** instances auto-size to your VRAM (≈2 on a T4). Push a T4 with
`--instances 3`, or `--precision fp16` for ~2× speed / half VRAM (preview the
quality first). `--voices male|female` and `--male-pct N` control the speakers.

In [ ]:
!python -m ghana_tts_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --hours 0.2 --name {NAME} --formats {FORMATS} --sample-rate {SR} --precision {PRECISION}
# tip: add  --instances 3  (push a T4) or  --precision fp16  (faster) once quality looks good
# from your own sentences instead:  --text-file {TEXT_FILE}  (drop --dataset/--text-column)

## Inspect the output

In [ ]:
import json, glob
from IPython.display import Audio, display

rows = [json.loads(l) for l in open(f'data/{NAME}/manifest.jsonl', encoding='utf-8')]
print(f'{len(rows)} clips, {sum(r["duration"] for r in rows)/3600:.3f} h')
for r in rows[:3]:
    print(f"\n[{r['gender']}] {r['duration']}s  {r['text'][:90]}")
    display(Audio(f"data/{NAME}/{r['file']}"))

## Custom speaker reference
Upload your own WAV files (mono, 16 kHz) to the Colab runtime, then point
`--speaker-dir` at the folder containing `male.wav`+`male.txt` and/or
`female.wav`+`female.txt`. Or override individually:

```bash
!python -m ghana_tts_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --hours 0.2 --name {NAME} --formats {FORMATS} \
    --speaker-male /content/my_voice.wav --speaker-male-text "my prompt sentence"
```
See `--help` for all speaker options.

## (Optional) Push to your own HF dataset repo
```bash
!python -m ghana_tts_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --hours 0.2 --name {NAME} --formats {FORMATS} --sample-rate {SR} \
    --push your-username/my-synth --token $HF_TOKEN
```
Full options: https://github.com/GhanaNLP/ghana-tts-datagen